In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow nltk transformers datasets accelerate torch

In [ ]:
import os
import re
import random
import inspect
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

TensorFlow version: 2.20.0
PyTorch version: 2.10.0+cu128
CUDA available: True


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving CEAS_08_cleaned.csv to CEAS_08_cleaned.csv
Saving machinewars_filtered_emails.json to machinewars_filtered_emails.json
Saving Nazario_cleaned.csv to Nazario_cleaned.csv
Saving Nigerian_Fraud_cleaned.csv to Nigerian_Fraud_cleaned.csv
Saving SpamAssasin_cleaned.csv to SpamAssasin_cleaned.csv


In [ ]:
import pandas as pd
import json
from pathlib import Path


# ---------------------------------
# 1. Helpers
# ---------------------------------
def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def build_text_subject_body(row):
    subject = safe_str(row.get("subject", ""))
    body = safe_str(row.get("body", ""))
    return f"{subject}\n\n{body}".strip()


# ---------------------------------
# 2. Label handling
# ---------------------------------
def normalize_machinewars_label(label, spam_as_phishing=False):
    """
    MachineWars:
      Phishing -> 1
      Legitimate/Valid/Ham -> 0
      Spam -> 1 if spam_as_phishing=True else excluded
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    if label == "spam":
        return 1 if spam_as_phishing else None

    return None


def normalize_test_label(label):
    """
    For CEAS-style test sets.
    Spam is excluded here unless you explicitly want otherwise.
    """
    if pd.isna(label):
        return None

    label = str(label).strip().lower()

    if label == "phishing":
        return 1

    if label in {"legitimate", "valid", "ham", "benign", "safe"}:
        return 0

    return None


# ---------------------------------
# 3. MachineWars loader
# ---------------------------------
def load_machinewars(json_path_or_list, spam_as_phishing=False, dataset_name="machinewars"):
    """
    Expected MachineWars fields:
      sender, subject, body, type, url
    """
    if isinstance(json_path_or_list, (str, Path)):
        with open(json_path_or_list, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = json_path_or_list

    df = pd.DataFrame(data).copy()

    # rename type -> label
    if "type" in df.columns:
        df = df.rename(columns={"type": "label"})

    # ensure required columns exist
    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    # keep optional columns if they exist
    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    # flatten url list to string
    df["url"] = df["url"].apply(
        lambda x: " | ".join(x) if isinstance(x, list) else safe_str(x)
    )

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(
        lambda x: normalize_machinewars_label(x, spam_as_phishing=spam_as_phishing)
    )

    # drop excluded rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)

    # normalized binary label name
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})

    # only subject + body for model text
    df["text"] = df.apply(build_text_subject_body, axis=1)

    # final schema
    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 4. CEAS-style test loader
# ---------------------------------
def load_ceas_style_csv(csv_path, dataset_name=None):
    """
    Assumes CEAS-style columns similar to:
      subject, body, label
    """
    csv_path = Path(csv_path)
    if dataset_name is None:
        dataset_name = csv_path.stem

    df = pd.read_csv(csv_path).copy()

    rename_map = {
        "Subject": "subject",
        "Body": "body",
        "Label": "label",
        "type": "label",
    }
    df = df.rename(columns=rename_map)

    for col in ["subject", "body", "label"]:
        if col not in df.columns:
            df[col] = ""

    if "sender" not in df.columns:
        df["sender"] = ""

    if "url" not in df.columns:
        df["url"] = ""

    df["dataset"] = dataset_name
    df["label_raw"] = df["label"].astype(str).str.strip().str.lower()
    df["label_id"] = df["label_raw"].apply(normalize_test_label)

    # drop non-binary rows
    df = df[df["label_id"].notna()].copy()
    df["label_id"] = df["label_id"].astype(int)
    df["label"] = df["label_id"].map({0: "legitimate", 1: "phishing"})
    df["text"] = df.apply(build_text_subject_body, axis=1)

    df = df[
        ["dataset", "sender", "subject", "body", "url", "label_raw", "label", "label_id", "text"]
    ]

    return df


# ---------------------------------
# 5. Create the two MachineWars versions
# ---------------------------------
machinewars_path = "machinewars_filtered_emails.json"

# Version A: spam merged into phishing
machinewars_spam_as_phishing_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=True,
    dataset_name="machinewars"
)

# Version B: spam removed
machinewars_no_spam_df = load_machinewars(
    machinewars_path,
    spam_as_phishing=False,
    dataset_name="machinewars"
)

print("MachineWars: spam merged into phishing")
print(machinewars_spam_as_phishing_df["label"].value_counts())
print(machinewars_spam_as_phishing_df.head(3))

print("\nMachineWars: spam removed")
print(machinewars_no_spam_df["label"].value_counts())
print(machinewars_no_spam_df.head(3))


# ---------------------------------
# 6. Load the 4 CEAS-style test datasets
# ---------------------------------
test_paths = [
    "CEAS_08_cleaned.csv",
    "Nazario_cleaned.csv",
    "Nigerian_Fraud_cleaned.csv",
    "SpamAssasin_cleaned.csv",
]

test_dfs = [load_ceas_style_csv(p) for p in test_paths]

for i, df in enumerate(test_dfs, 1):
    print(f"\nTest dataset {i}:")
    print(df["label"].value_counts())
    print(df.head(2))

MachineWars: spam merged into phishing
label
phishing      13200
legitimate     6600
Name: count, dtype: int64
       dataset                                             sender  \
0  machinewars      Dropbox Security <noreply@dropbox-secure.net>   
1  machinewars  Google Drive Security <security-alert@google-d...   
2  machinewars  Microsoft OneDrive Security <noreply@microsoft...   

                                             subject  \
0  Unusual Sign-in Activity Detected on Your Drop...   
1  Security Alert: New Sign-in to Your Google Dri...   
2  Important Security Notification Regarding Your...   

                                                body  \
0  Dear User,\n\nWe've detected an unusual sign-i...   
1  Google Drive Security Alert\n\nWe've noticed a...   
2  Hello sarah.smith@gmail.com,\n\nThis is an aut...   

                                                 url label_raw     label  \
0         https://dropbox-security.co/account/review  phishing  phishing   
1  https:/

In [ ]:
train_df, val_df = train_test_split(
    machinewars_no_spam_df,
    test_size=0.2,
    random_state=SEED,
    stratify=machinewars_no_spam_df["label_id"]
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(
    train_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

val_ds = Dataset.from_pandas(
    val_df[["text", "label_id"]]
    .rename(columns={"label_id": "labels"})
    .reset_index(drop=True)
)

In [ ]:
def compute_binary_metrics(y_true, y_pred, y_prob):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )
    acc = accuracy_score(y_true, y_pred)

    try:
        roc_auc = roc_auc_score(y_true, y_prob)
    except Exception:
        roc_auc = float("nan")

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
    }
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    # If model returns tuple-like predictions, keep logits only.
    if isinstance(logits, tuple):
        logits = logits[0]

    y_true = labels
    y_pred = np.argmax(logits, axis=-1)

    # Convert logits to probability for class 1.
    exp_logits = np.exp(logits - np.max(logits, axis=1, keepdims=True))
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    y_prob = probs[:, 1]

    return compute_binary_metrics(y_true, y_pred, y_prob)

In [ ]:
X_train = train_df["text"].astype(str).tolist()
y_train = train_df["label_id"].astype(int).values

X_val = val_df["text"].astype(str).tolist()
y_val = val_df["label_id"].astype(int).values


DISTILBERT_MODEL_NAME = "distilbert-base-uncased"
DISTILBERT_MAX_LENGTH = 512

distilbert_tokenizer = AutoTokenizer.from_pretrained(DISTILBERT_MODEL_NAME)

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=2,
)


train_texts = [str(x) for x in X_train]
eval_texts = [str(x) for x in X_val]

train_labels = [int(y) for y in y_train]
eval_labels = [int(y) for y in y_val]

train_dataset = Dataset.from_dict({
    "text": train_texts,
    "labels": train_labels,
})

eval_dataset = Dataset.from_dict({
    "text": eval_texts,
    "labels": eval_labels,
})

def tokenize_distilbert_batch(batch):
    return distilbert_tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=DISTILBERT_MAX_LENGTH,
    )

train_dataset = train_dataset.map(
    tokenize_distilbert_batch,
    batched=True,
    remove_columns=["text"],
)

eval_dataset = eval_dataset.map(
    tokenize_distilbert_batch,
    batched=True,
    remove_columns=["text"],
)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

eval_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"],
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/10684 [00:00<?, ? examples/s]

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
DISTILBERT_BATCH_SIZE = 32
DISTILBERT_EPOCHS = 3

# DistilBERT has no token_type_ids; truncation is important because emails can be long.


def tokenize_distilbert_batch(batch):
    return distilbert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=DISTILBERT_MAX_LENGTH,
    )


train_ds_tok = train_ds.map(tokenize_distilbert_batch, batched=True)
val_ds_tok = val_ds.map(tokenize_distilbert_batch, batched=True)

# Keep only tensors needed by Trainer.
train_ds_tok = train_ds_tok.remove_columns([c for c in train_ds_tok.column_names if c not in {"input_ids", "attention_mask", "labels"}])
val_ds_tok = val_ds_tok.remove_columns([c for c in val_ds_tok.column_names if c not in {"input_ids", "attention_mask", "labels"}])

id2label = {0: "legitimate", 1: "phishing"}
label2id = {v: k for k, v in id2label.items()}

distilbert_model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorWithPadding(tokenizer=distilbert_tokenizer)

# Keep the SVM's class_weight="balanced" behavior by weighting cross-entropy.
class_counts = np.bincount(y_train, minlength=2)
class_weights = len(y_train) / (2.0 * np.maximum(class_counts, 1))
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

classes = np.array([0, 1])

class_weights_np = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_labels),
)

class_weights = torch.tensor(class_weights_np, dtype=torch.float)
print("Class weights:", class_weights)


class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weights = self.class_weights.to(outputs.logits.device) if self.class_weights is not None else None
        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(outputs.logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_distilbert_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
    preds = (probs >= 0.5).astype(int)
    return compute_binary_metrics(labels, preds, probs)



# Transformers renamed evaluation_strategy to eval_strategy in newer releases.
# This small compatibility shim lets the notebook run on both old and new Colab images.
import inspect
training_args_kwargs = dict(
    output_dir="/content/distilbert_phishing_model",
    learning_rate=2e-5,
    per_device_train_batch_size=DISTILBERT_BATCH_SIZE,
    per_device_eval_batch_size=DISTILBERT_BATCH_SIZE,
    num_train_epochs=DISTILBERT_EPOCHS,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    training_args_kwargs["eval_strategy"] = "epoch"
else:
    training_args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_args_kwargs)

distilbert_trainer = WeightedTrainer(
    model=distilbert_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

distilbert_trainer.train()

print("DistilBERT training complete.")


Map:   0%|          | 0/10684 [00:00<?, ? examples/s]

Map:   0%|          | 0/2672 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Class weights: tensor([1.0117, 0.9885])


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.073667,0.042741,0.984281,0.978801,0.990385,0.984559,0.998989
2,0.016841,0.038657,0.989147,0.996995,0.981509,0.989191,0.999622
3,0.007455,0.022587,0.994760,0.994822,0.994822,0.994822,0.999829


DistilBERT training complete.


In [ ]:
def distilbert_predict_one(text):
    pred, prob = distilbert_batch_predict([text], batch_size=1)
    return int(pred[0]), float(prob[0])


def distilbert_batch_predict(texts, batch_size=32):
    texts = [str(t) for t in texts]
    if len(texts) == 0:
        return np.array([], dtype=int), np.array([], dtype=float)

    distilbert_model.eval()
    all_probs = []
    device = distilbert_model.device

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = distilbert_tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=DISTILBERT_MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            logits = distilbert_model(**encoded).logits
            probs = torch.softmax(logits, dim=-1)[:, 1]

        all_probs.append(probs.detach().cpu().numpy())

    probs = np.concatenate(all_probs)
    preds = (probs >= 0.5).astype(int)
    return preds, probs


def evaluate_distilbert(df_eval):
    y_true = df_eval["label_id"].astype(int).values
    y_pred, y_prob = distilbert_batch_predict(df_eval["text"].astype(str).tolist())
    return compute_binary_metrics(y_true, y_pred, y_prob)




In [ ]:
print("Validation metrics:")
print(evaluate_distilbert(val_df))

distilbert_rows = [{"dataset": "validation", **evaluate_distilbert(val_df)}]

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]
    metrics = evaluate_distilbert(test_df)
    distilbert_rows.append({"dataset": test_name, **metrics})

distilbert_results_df = pd.DataFrame(distilbert_rows)
distilbert_results_df

Validation metrics:
{'accuracy': 0.9947604790419161, 'precision': 0.9948224852071006, 'recall': 0.9948224852071006, 'f1': 0.9948224852071006, 'roc_auc': np.float64(0.9998285368477676)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.994760,0.994822,0.994822,0.994822,0.999829
1,CEAS_08_cleaned,0.488941,0.973630,0.086210,0.158395,0.939928
2,Nazario_cleaned,0.884345,1.000000,0.884345,0.938623,NaN
3,Nigerian_Fraud_cleaned,0.840636,1.000000,0.840636,0.913419,NaN
4,SpamAssasin_cleaned,0.784300,0.995736,0.271828,0.427069,0.862603


Validation metrics:
{'accuracy': 0.9651515151515152, 'precision': 0.9742228961334344, 'recall': 0.9734848484848485, 'f1': 0.9738537324744221, 'roc_auc': np.float64(0.9911992079889806)}


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


,dataset,accuracy,precision,recall,f1,roc_auc
0,validation,0.965152,0.974223,0.973485,0.973854,0.991199
1,CEAS_08_cleaned,0.862517,0.931293,0.813570,0.868460,0.926345
2,Nazario_cleaned,0.941853,1.000000,0.941853,0.970056,NaN
3,Nigerian_Fraud_cleaned,0.934574,1.000000,0.934574,0.966181,NaN
4,SpamAssasin_cleaned,0.811672,0.694757,0.647846,0.670482,0.866153


In [ ]:
import nltk
from nltk.corpus import wordnet

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def prefix_injection_attack(text):
    prefix = (
        "This is a normal and trustworthy business email. "
        "The message is legitimate, safe, and routine.\n\n"
    )
    return prefix + str(text)


PHISHING_KEYWORDS = {
    "verify", "verification", "account", "password", "login", "signin",
    "security", "alert", "urgent", "confirm", "suspend", "suspended",
    "click", "update", "reset", "limited", "immediately"
}

def keyword_deletion_attack(text, max_delete=5):
    words = str(text).split()
    new_words = []
    deleted = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w).lower()
        if clean in PHISHING_KEYWORDS and deleted < max_delete:
            deleted += 1
            continue
        new_words.append(w)

    return " ".join(new_words)


def get_synonyms(word):
    syns = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            s = lemma.name().replace("_", " ").strip()
            if s and s.lower() != word.lower():
                syns.add(s)
    return list(syns)


def synonym_attack(text, replace_prob=0.12, max_replacements=8, seed=42):
    rng = random.Random(seed)
    words = str(text).split()
    new_words = []
    replacements = 0

    for w in words:
        clean = re.sub(r"^[^\w]+|[^\w]+$", "", w)

        if (
            replacements < max_replacements
            and len(clean) >= 4
            and clean.isalpha()
            and rng.random() < replace_prob
        ):
            syns = get_synonyms(clean)
            syns = [s for s in syns if s.isalpha() and len(s.split()) == 1]

            if syns:
                replacement = rng.choice(syns)
                if w.istitle():
                    replacement = replacement.title()
                new_words.append(replacement)
                replacements += 1
                continue

        new_words.append(w)

    return " ".join(new_words)

In [ ]:
predict_one = distilbert_predict_one
batch_predict = distilbert_batch_predict
ACTIVE_MODEL_NAME = "distilbert"
print("Active model:", ACTIVE_MODEL_NAME)

Active model: distilbert


In [ ]:
def evaluate_attack_common(df_eval, attack_name, attack_fn):
    attacked_texts = [attack_fn(t) for t in df_eval["text"].astype(str).tolist()]
    y_true = df_eval["label_id"].astype(int).values

    y_pred, y_prob = batch_predict(attacked_texts)

    return {
        "attack": attack_name,
        "n_samples": len(df_eval),
        **compute_binary_metrics(y_true, y_pred, y_prob)
    }

In [ ]:
attack_rows_val = []

attack_rows_val.append(evaluate_attack_common(val_df, "clean", lambda x: x))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_prefix", benign_prefix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "benign_suffix", benign_suffix_attack))
attack_rows_val.append(evaluate_attack_common(val_df, "contradiction", contradiction_attack))
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "synonym_attack",
        lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
    )
)
attack_rows_val.append(
    evaluate_attack_common(
        val_df,
        "keyword_deletion",
        lambda x: keyword_deletion_attack(x, max_delete=5)
    )
)
attack_rows_val.append(evaluate_attack_common(val_df, "prefix_injection", prefix_injection_attack))

attack_results_val_df = pd.DataFrame(attack_rows_val).sort_values("f1", ascending=False)
attack_results_val_df

,attack,n_samples,accuracy,precision,recall,f1,roc_auc
0,clean,2672,0.994760,0.994822,0.994822,0.994822,0.999829
2,benign_suffix,2672,0.994386,0.994087,0.994822,0.994455,0.999822
5,keyword_deletion,2672,0.994386,0.995552,0.993343,0.994447,0.999809
4,synonym_attack,2672,0.993638,0.994811,0.992604,0.993706,0.999762
3,contradiction,2672,0.991766,0.987537,0.996302,0.991900,0.999769
6,prefix_injection,2672,0.991018,0.986091,0.996302,0.991170,0.999783
1,benign_prefix,2672,0.985030,0.973304,0.997781,0.985391,0.999798


In [ ]:
all_attack_tables = {}

for test_df in test_dfs:
    test_name = test_df["dataset"].iloc[0]

    rows = []
    rows.append(evaluate_attack_common(test_df, "clean", lambda x: x))
    rows.append(evaluate_attack_common(test_df, "benign_prefix", benign_prefix_attack))
    rows.append(evaluate_attack_common(test_df, "benign_suffix", benign_suffix_attack))
    rows.append(evaluate_attack_common(test_df, "contradiction", contradiction_attack))
    rows.append(
        evaluate_attack_common(
            test_df,
            "synonym_attack",
            lambda x: synonym_attack(x, replace_prob=0.12, max_replacements=8, seed=42)
        )
    )
    rows.append(
        evaluate_attack_common(
            test_df,
            "keyword_deletion",
            lambda x: keyword_deletion_attack(x, max_delete=5)
        )
    )
    rows.append(evaluate_attack_common(test_df, "prefix_injection", prefix_injection_attack))

    result_df = pd.DataFrame(rows).sort_values("f1", ascending=False)
    all_attack_tables[test_name] = result_df

    print(f"\n=== {ACTIVE_MODEL_NAME} | {test_name} ===")
    print(result_df)


=== distilbert | CEAS_08_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix      39154  0.599019   0.942252  0.299560  0.454596   
3     contradiction      39154  0.550570   0.960512  0.202683  0.334732   
6  prefix_injection      39154  0.538412   0.964048  0.179242  0.302282   
2     benign_suffix      39154  0.523676   0.973591  0.150215  0.260273   
0             clean      39154  0.488941   0.973630  0.086210  0.158395   
5  keyword_deletion      39154  0.487179   0.977260  0.082639  0.152391   
4    synonym_attack      39154  0.484472   0.975330  0.077832  0.144159   

    roc_auc  
1  0.907469  
3  0.931800  
6  0.928027  
2  0.936004  
0  0.939928  
5  0.940160  
4  0.939488  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== distilbert | Nazario_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       1565  0.950160        1.0  0.950160  0.974443   
3     contradiction       1565  0.932907        1.0  0.932907  0.965289   
6  prefix_injection       1565  0.925240        1.0  0.925240  0.961168   
2     benign_suffix       1565  0.897764        1.0  0.897764  0.946128   
0             clean       1565  0.884345        1.0  0.884345  0.938623   
5  keyword_deletion       1565  0.860703        1.0  0.860703  0.925137   
4    synonym_attack       1565  0.856230        1.0  0.856230  0.922547   

   roc_auc  
1      NaN  
3      NaN  
6      NaN  
2      NaN  
0      NaN  
5      NaN  
4      NaN  


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist


=== distilbert | Nigerian_Fraud_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
3     contradiction       3332  0.882053        1.0  0.882053  0.937331   
1     benign_prefix       3332  0.879352        1.0  0.879352  0.935803   
6  prefix_injection       3332  0.860444        1.0  0.860444  0.924988   
2     benign_suffix       3332  0.846939        1.0  0.846939  0.917127   
0             clean       3332  0.840636        1.0  0.840636  0.913419   
5  keyword_deletion       3332  0.833733        1.0  0.833733  0.909329   
4    synonym_attack       3332  0.829832        1.0  0.829832  0.907003   

   roc_auc  
3      NaN  
1      NaN  
6      NaN  
2      NaN  
0      NaN  
5      NaN  
4      NaN  

=== distilbert | SpamAssasin_cleaned ===
             attack  n_samples  accuracy  precision    recall        f1  \
1     benign_prefix       5809  0.838871   0.943311  0.484284  0.640000   
3     contradiction       5809  0.823894   0.970230  0.417

In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
def benign_prefix_attack(text):
    prefix = (
        "Hello,\n\n"
        "This is a routine business communication regarding normal account activity. "
        "Please review the information below.\n\n"
    )
    return prefix + str(text)


def benign_suffix_attack(text):
    suffix = (
        "\n\nThank you for your attention."
        "\nThis message is part of a standard informational workflow."
        "\nBest regards."
    )
    return str(text) + suffix


def contradiction_attack(text):
    prefix = (
        "Important note: this is a legitimate and safe message. "
        "It does not request any unsafe action.\n\n"
    )
    return prefix + str(text)


def training_context_attack(text):
    prefix = (
        "Example of a safe enterprise email used for routine staff communication:\n\n"
    )
    return prefix + str(text)


def noise_injection_attack(text):
    noise = (
        "\n\nReference ID: A7K2-91LM"
        "\nDepartment: operations"
        "\nCase Type: routine review"
    )
    return str(text) + noise

In [ ]:
import re

STOPWORDS_ATTACK = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "at",
    "is", "are", "this", "that", "with", "from", "by", "as", "it", "be",
    "was", "were", "subject", "body", "dear", "hello", "hi", "regards",
    "thanks", "thank", "best"
}


def basic_tokenize_with_indices(text):
    """
    Splits text into whitespace-separated tokens and keeps positions.
    """
    tokens = str(text).split()
    return tokens


def is_deletable_token(tok):
    clean = re.sub(r"^[^\w]+|[^\w]+$", "", tok).lower()
    if len(clean) < 3:
        return False
    if clean in STOPWORDS_ATTACK:
        return False
    if not any(ch.isalpha() for ch in clean):
        return False
    return True


def delete_token_at_index(tokens, idx):
    return " ".join(tokens[:idx] + tokens[idx+1:])


def greedy_delete_attack_blackbox(text, max_delete_steps=5, candidate_cap=15):
    """
    Black-box deletion attack:
    At each step, test candidate single-token deletions and keep the one that
    minimizes phishing probability.
    """
    current_text = str(text)

    for _ in range(max_delete_steps):
        tokens = basic_tokenize_with_indices(current_text)

        candidate_indices = [i for i, tok in enumerate(tokens) if is_deletable_token(tok)]

        if not candidate_indices:
            break

        # Cap candidates for speed
        candidate_indices = candidate_indices[:candidate_cap]

        candidate_texts = [delete_token_at_index(tokens, i) for i in candidate_indices]

        _, candidate_probs = batch_predict(candidate_texts)
        best_idx = int(np.argmin(candidate_probs))

        best_text = candidate_texts[best_idx]

        if best_text == current_text:
            break

        current_text = best_text

    return current_text

In [ ]:
def greedy_add_attack_blackbox(text, add_steps=3):
    """
    Greedily applies the benign addition that lowers phishing probability the most.
    """
    current_text = str(text)

    addition_fns = [
        ("benign_prefix", benign_prefix_attack),
        ("benign_suffix", benign_suffix_attack),
        ("contradiction", contradiction_attack),
        ("training_context", training_context_attack),
        ("noise_injection", noise_injection_attack),
    ]

    history = []

    for step in range(1, add_steps + 1):
        candidate_names = []
        candidate_texts = []

        for attack_name, attack_fn in addition_fns:
            candidate_names.append(attack_name)
            candidate_texts.append(attack_fn(current_text))

        candidate_preds, candidate_probs = batch_predict(candidate_texts)

        best_idx = int(np.argmin(candidate_probs))
        current_text = candidate_texts[best_idx]

        history.append({
            "step": step,
            "attack_name": candidate_names[best_idx],
            "pred": int(candidate_preds[best_idx]),
            "phishing_prob": float(candidate_probs[best_idx]),
        })

    return current_text, history

In [ ]:
def add_only_attack(text, add_steps=3):
    attacked_text, _ = greedy_add_attack_blackbox(text, add_steps=add_steps)
    return attacked_text


def delete_only_attack(text, delete_steps=5):
    attacked_text = greedy_delete_attack_blackbox(text, max_delete_steps=delete_steps)
    return attacked_text


def hybrid_add_then_delete_attack_blackbox(text, add_steps=3, delete_steps=5):
    """
    First greedy additions, then greedy deletions.
    """
    current_text, add_history = greedy_add_attack_blackbox(text, add_steps=add_steps)
    current_text = greedy_delete_attack_blackbox(current_text, max_delete_steps=delete_steps)
    return current_text

In [ ]:
def evaluate_attack_detailed(df_eval, attack_name, attack_fn, show_progress=True):
    y_true = df_eval["label_id"].astype(int).tolist()
    texts = df_eval["text"].astype(str).tolist()

    orig_pred, orig_prob = batch_predict(texts)

    attacked_texts = []
    iterator = texts
    if show_progress:
        iterator = tqdm(texts, total=len(texts), desc=f"{ACTIVE_MODEL_NAME} | {attack_name}")

    for text in iterator:
        attacked_texts.append(attack_fn(text))

    new_pred, new_prob = batch_predict(attacked_texts)

    metrics = compute_binary_metrics(y_true, new_pred, new_prob)

    details_df = pd.DataFrame({
        "text": texts,
        "label_id": y_true,
        "orig_pred": orig_pred,
        "orig_prob": orig_prob,
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["orig_pred"] != details_df["new_pred"]
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    summary = {
        "attack": attack_name,
        "n_samples": len(df_eval),
        "flip_rate": float(details_df["flipped"].mean()),
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
        **metrics,
    }

    return summary, details_df

In [ ]:
def evaluate_evasion_on_phishing(df_eval, attack_name, attack_fn, show_progress=True):
    """
    Restrict evaluation to:
    - true phishing samples
    - originally correct phishing predictions

    Reports attack success rate (ASR).
    """
    df_local = df_eval.copy()
    df_local = df_local[df_local["label_id"].astype(int) == 1].copy()

    if len(df_local) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": 0,
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    texts = df_local["text"].astype(str).tolist()
    orig_pred, orig_prob = batch_predict(texts)

    df_local["orig_pred"] = orig_pred
    df_local["orig_prob"] = orig_prob

    df_attack = df_local[df_local["orig_pred"] == 1].copy()

    if len(df_attack) == 0:
        return {
            "attack": attack_name,
            "n_true_phishing": len(df_local),
            "n_orig_correct_phishing": 0,
            "n_flipped": 0,
            "attack_success_rate": float("nan"),
            "robust_recall_on_orig_correct_phishing": float("nan"),
            "avg_prob_drop": float("nan"),
        }, pd.DataFrame()

    attacked_texts = []
    iterator = df_attack["text"].astype(str).tolist()
    if show_progress:
        iterator = tqdm(iterator, total=len(df_attack), desc=f"{ACTIVE_MODEL_NAME} | {attack_name} phishing")

    for text in iterator:
        attacked_texts.append(attack_fn(text))

    new_pred, new_prob = batch_predict(attacked_texts)

    details_df = pd.DataFrame({
        "text": df_attack["text"].astype(str).tolist(),
        "label_id": df_attack["label_id"].astype(int).tolist(),
        "orig_pred": df_attack["orig_pred"].tolist(),
        "orig_prob": df_attack["orig_prob"].tolist(),
        "attacked_text": attacked_texts,
        "new_pred": new_pred,
        "new_prob": new_prob,
    })

    details_df["flipped"] = details_df["new_pred"] != 1
    details_df["prob_drop"] = details_df["orig_prob"] - details_df["new_prob"]

    n_orig_correct = len(details_df)
    n_flipped = int(details_df["flipped"].sum())
    asr = n_flipped / n_orig_correct
    robust_recall = 1.0 - asr

    summary = {
        "attack": attack_name,
        "n_true_phishing": len(df_local),
        "n_orig_correct_phishing": n_orig_correct,
        "n_flipped": n_flipped,
        "attack_success_rate": asr,
        "robust_recall_on_orig_correct_phishing": robust_recall,
        "avg_prob_drop": float(details_df["prob_drop"].mean()),
    }

    return summary, details_df

In [ ]:
val_attack_summaries = []
val_attack_details = {}

attack_specs = [
    ("add_only_add3", lambda x: add_only_attack(x, add_steps=3)),
    ("delete_only_del5", lambda x: delete_only_attack(x, delete_steps=5)),
    ("hybrid_add3_delete5", lambda x: hybrid_add_then_delete_attack_blackbox(x, add_steps=3, delete_steps=5)),
]

In [ ]:
for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_attack_detailed(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_attack_summaries.append(summary)
    val_attack_details[attack_name] = details

val_attack_results_df = pd.DataFrame(val_attack_summaries).sort_values("f1", ascending=False)
val_attack_results_df

distilbert | add_only_add3:   0%|          | 0/2672 [00:00<?, ?it/s]

distilbert | delete_only_del5:   0%|          | 0/2672 [00:00<?, ?it/s]

distilbert | hybrid_add3_delete5:   0%|          | 0/2672 [00:00<?, ?it/s]

,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,add_only_add3,2672,0.007111,0.007785,0.988398,0.995499,0.981509,0.988454,0.999674
1,delete_only_del5,2672,0.007859,0.007472,0.988398,0.996243,0.980769,0.988446,0.999593
2,hybrid_add3_delete5,2672,0.013473,0.014341,0.983533,0.996960,0.970414,0.983508,0.999501


In [ ]:
val_evasion_summaries = []
val_evasion_details = {}

for attack_name, attack_fn in attack_specs:
    summary, details = evaluate_evasion_on_phishing(
        val_df,
        attack_name,
        attack_fn,
        show_progress=True
    )
    val_evasion_summaries.append(summary)
    val_evasion_details[attack_name] = details

val_evasion_results_df = pd.DataFrame(val_evasion_summaries).sort_values("attack_success_rate", ascending=False)
val_evasion_results_df

distilbert | add_only_add3 phishing:   0%|          | 0/1345 [00:00<?, ?it/s]

distilbert | delete_only_del5 phishing:   0%|          | 0/1345 [00:00<?, ?it/s]

distilbert | hybrid_add3_delete5 phishing:   0%|          | 0/1345 [00:00<?, ?it/s]

,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
2,hybrid_add3_delete5,1352,1345,33,0.024535,0.975465,0.025046
1,delete_only_del5,1352,1345,19,0.014126,0.985874,0.012633
0,add_only_add3,1352,1345,18,0.013383,0.986617,0.013377


In [ ]:

all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)



Starting dataset: CEAS_08_cleaned
Rows: 39154

Running attack: add_only_add3


distilbert | add_only_add3:   0%|          | 0/39154 [00:00<?, ?it/s]

distilbert | add_only_add3 phishing:   0%|          | 0/1883 [00:00<?, ?it/s]


Attack metrics:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,CEAS_08_cleaned,add_only_add3,39154,0.034786,0.037327,0.455739,0.961806,0.025364,0.049425,0.908241


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,Nazario_cleaned,delete_only_del5,1565,1384,238,0.171965,0.828035,0.162309



Running attack: hybrid_add3_delete5


distilbert | hybrid_add3_delete5:   0%|          | 0/1565 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


distilbert | hybrid_add3_delete5 phishing:   0%|          | 0/1384 [00:00<?, ?it/s]

In [ ]:

all_test_attack_results = {}
all_test_evasion_results = {}

for test_df in test_dfs:
    dataset_name = test_df["dataset"].iloc[0]

    if dataset_name == "CEAS_08_cleaned":
        print(f"\nSkipping dataset: {dataset_name}")
        continue

    print(f"\n\n==============================")
    print(f"Starting dataset: {dataset_name}")
    print(f"Rows: {len(test_df)}")
    print(f"==============================")

    attack_summaries = []
    evasion_summaries = []

    for attack_name, attack_fn in attack_specs:
        print(f"\nRunning attack: {attack_name}")

        summary_attack, _ = evaluate_attack_detailed(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        attack_summaries.append({"dataset": dataset_name, **summary_attack})

        summary_evasion, _ = evaluate_evasion_on_phishing(
            test_df,
            attack_name,
            attack_fn,
            show_progress=True
        )
        evasion_summaries.append({"dataset": dataset_name, **summary_evasion})

        print("\nAttack metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_attack}]))

        print("\nPhishing evasion metrics:")
        display(pd.DataFrame([{"dataset": dataset_name, **summary_evasion}]))

    dataset_attack_df = pd.DataFrame(attack_summaries)
    dataset_evasion_df = pd.DataFrame(evasion_summaries)

    all_test_attack_results[dataset_name] = dataset_attack_df
    all_test_evasion_results[dataset_name] = dataset_evasion_df

    print(f"\nFinished dataset: {dataset_name}")

    print("\nAll attack metrics for this dataset:")
    display(dataset_attack_df)

    print("\nAll phishing evasion metrics for this dataset:")
    display(dataset_evasion_df)


Skipping dataset: CEAS_08_cleaned


Starting dataset: Nazario_cleaned
Rows: 1565

Running attack: add_only_add3


distilbert | add_only_add3:   0%|          | 0/1565 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


distilbert | add_only_add3 phishing:   0%|          | 0/1384 [00:00<?, ?it/s]


Attack metrics:


...

,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,hybrid_add3_delete5,1718,467,283,0.605996,0.394004,0.513593



Finished dataset: SpamAssasin_cleaned

All attack metrics for this dataset:


,dataset,attack,n_samples,flip_rate,avg_prob_drop,accuracy,precision,recall,f1,roc_auc
0,SpamAssasin_cleaned,add_only_add3,5809,0.031847,0.032093,0.753486,0.996528,0.167055,0.286142,0.870622
1,SpamAssasin_cleaned,delete_only_del5,5809,0.032536,0.030886,0.752797,1.000000,0.164144,0.282000,0.855502
2,SpamAssasin_cleaned,hybrid_add3_delete5,5809,0.049062,0.049597,0.735927,1.000000,0.107101,0.193481,0.869848



All phishing evasion metrics for this dataset:


,dataset,attack,n_true_phishing,n_orig_correct_phishing,n_flipped,attack_success_rate,robust_recall_on_orig_correct_phishing,avg_prob_drop
0,SpamAssasin_cleaned,add_only_add3,1718,467,182,0.389722,0.610278,0.315738
1,SpamAssasin_cleaned,delete_only_del5,1718,467,186,0.398287,0.601713,0.304418
2,SpamAssasin_cleaned,hybrid_add3_delete5,1718,467,283,0.605996,0.394004,0.513593


In [ ]:
combined_attack_df = pd.concat(all_test_attack_results.values(), ignore_index=True)
combined_evasion_df = pd.concat(all_test_evasion_results.values(), ignore_index=True)

print("Combined attack results:")
print(combined_attack_df)

print("\nCombined evasion results:")
print(combined_evasion_df)

os.makedirs("/content/results", exist_ok=True)

distilbert_results_df.to_csv("/content/results/distilbert_clean_results.csv", index=False)
val_attack_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_metrics.csv", index=False)
val_evasion_results_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_val_add_delete_hybrid_evasion.csv", index=False)
combined_attack_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_metrics.csv", index=False)
combined_evasion_df.to_csv(f"/content/results/{ACTIVE_MODEL_NAME}_test_add_delete_hybrid_evasion.csv", index=False)

print("Saved DistilBERT result files to /content/results")